In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

print("Loading Data...")

# =========================
# LOAD DATA
# =========================

nav = pd.read_csv("../data/raw/02_nav_history.csv")
benchmark = pd.read_csv("../data/raw/10_benchmark_indices.csv")
performance = pd.read_csv("../data/raw/07_scheme_performance.csv")

nav["date"] = pd.to_datetime(nav["date"])
benchmark["date"] = pd.to_datetime(benchmark["date"])

print("Data Loaded")

# =========================
# DAILY RETURNS
# =========================

nav = nav.sort_values(["amfi_code","date"])

nav["daily_return"] = (
    nav.groupby("amfi_code")["nav"]
       .pct_change()
)

print("Daily Returns Computed")

# =========================
# CAGR TABLE
# =========================

cagr_list = []

for fund in nav["amfi_code"].unique():

    temp = nav[nav["amfi_code"] == fund]

    start_nav = temp["nav"].iloc[0]
    end_nav = temp["nav"].iloc[-1]

    years = (
        (temp["date"].max() - temp["date"].min()).days
        / 365
    )

    cagr = ((end_nav/start_nav)**(1/years)-1)*100

    cagr_list.append([fund,cagr])

cagr_df = pd.DataFrame(
    cagr_list,
    columns=["amfi_code","cagr_pct"]
)

print("CAGR Done")

# =========================
# SHARPE RATIO
# =========================

RF = 0.065

sharpe_list = []

for fund in nav["amfi_code"].unique():

    r = nav[
        nav["amfi_code"]==fund
    ]["daily_return"].dropna()

    if len(r)==0:
        continue

    sharpe = (
        (r.mean()*252 - RF)
        /
        (r.std())
    ) * np.sqrt(252)

    sharpe_list.append([fund,sharpe])

sharpe_df = pd.DataFrame(
    sharpe_list,
    columns=["amfi_code","sharpe_ratio"]
)

print("Sharpe Done")

# =========================
# SORTINO RATIO
# =========================

sortino_list = []

for fund in nav["amfi_code"].unique():

    r = nav[
        nav["amfi_code"]==fund
    ]["daily_return"].dropna()

    downside = r[r < 0]

    if len(downside)==0:
        continue

    sortino = (
        (r.mean()*252 - RF)
        /
        downside.std()
    ) * np.sqrt(252)

    sortino_list.append([fund,sortino])

sortino_df = pd.DataFrame(
    sortino_list,
    columns=["amfi_code","sortino_ratio"]
)

print("Sortino Done")

# =========================
# ALPHA BETA
# =========================

nifty = benchmark[
    benchmark["index_name"].str.contains(
        "Nifty",
        case=False,
        na=False
    )
].copy()

nifty = nifty.sort_values("date")

nifty["benchmark_return"] = (
    nifty["close_value"].pct_change()
)

alpha_beta = []

for fund in nav["amfi_code"].unique():

    fund_df = nav[
        nav["amfi_code"]==fund
    ][["date","daily_return"]]

    merged = pd.merge(
        fund_df,
        nifty[["date","benchmark_return"]],
        on="date"
    ).dropna()

    if len(merged) < 30:
        continue

    slope, intercept, r, p, std = linregress(
        merged["benchmark_return"],
        merged["daily_return"]
    )

    alpha = intercept * 252
    beta = slope

    alpha_beta.append(
        [fund,alpha,beta]
    )

alpha_beta_df = pd.DataFrame(
    alpha_beta,
    columns=["amfi_code","alpha","beta"]
)

alpha_beta_df.to_csv(
    "../reports/alpha_beta.csv",
    index=False
)

print("Alpha Beta Saved")

# =========================
# MAX DRAWDOWN
# =========================

dd_list = []

for fund in nav["amfi_code"].unique():

    temp = nav[
        nav["amfi_code"]==fund
    ]

    running_max = temp["nav"].cummax()

    drawdown = (
        temp["nav"] /
        running_max
        - 1
    )

    dd_list.append([
        fund,
        drawdown.min()*100
    ])

dd_df = pd.DataFrame(
    dd_list,
    columns=["amfi_code","max_drawdown_pct"]
)

print("Drawdown Done")

# =========================
# FUND SCORECARD
# =========================

scorecard = (
    cagr_df
    .merge(sharpe_df,on="amfi_code")
    .merge(alpha_beta_df,on="amfi_code")
    .merge(dd_df,on="amfi_code")
)

scorecard["score"] = (
      scorecard["cagr_pct"].rank(pct=True)*30
    + scorecard["sharpe_ratio"].rank(pct=True)*25
    + scorecard["alpha"].rank(pct=True)*20
    + scorecard["max_drawdown_pct"].rank(
        ascending=False,
        pct=True
      )*25
)

scorecard = scorecard.sort_values(
    "score",
    ascending=False
)

scorecard.to_csv(
    "../reports/fund_scorecard.csv",
    index=False
)

print("Fund Scorecard Saved")

# =========================
# BENCHMARK COMPARISON
# =========================

top5 = scorecard.head(5)["amfi_code"]

plt.figure(figsize=(12,6))

for fund in top5:

    temp = nav[
        nav["amfi_code"]==fund
    ]

    plt.plot(
        temp["date"],
        temp["nav"],
        label=str(fund)
    )

plt.title(
    "Top 5 Funds Benchmark Comparison"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    "../reports/charts/benchmark_comparison.png"
)

plt.close()

print("Benchmark Chart Saved")

print("\nDAY 4 COMPLETED SUCCESSFULLY")

Loading Data...
Data Loaded
Daily Returns Computed
CAGR Done
Sharpe Done
Sortino Done
Alpha Beta Saved
Drawdown Done
Fund Scorecard Saved
Benchmark Chart Saved

DAY 4 COMPLETED SUCCESSFULLY
